### RAG pipeline - Data ingestion to vector DB pipeline


In [27]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Work Opportunity Tax Credit (WOTC).pdf
  ✓ Loaded 1 pages

Processing: saikumar_resume_java (6).pdf
  ✓ Loaded 2 pages

Processing: paper-drieve.pdf
  ✓ Loaded 6 pages

Total documents loaded: 9


: 

In [ ]:
all_pdf_documents

[Document(metadata={'producer': 'PDF-XChange Printer V6 (6.0 build 317.1) [Windows 10 Enterprise x64 (Build 19045)]', 'creator': 'PyPDF', 'creationdate': '2024-11-25T19:56:12-06:00', 'source': '../data/pdf/Work Opportunity Tax Credit (WOTC).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Work Opportunity Tax Credit (WOTC).pdf', 'file_type': 'pdf'}, page_content='Work Opportunity Tax Credits – KBR, Inc.\nKBR, Inc. participates in the Work Opportunity Tax Credit (WOTC) program, which ADP\nadministers on behalf of the company. Please follow the steps listed below to screen for\nthe WOTC program.  We appreciate your cooperation.\nApplicant Instructions\n· Open https://tcs.adp.com/kbrinc  or scan the QR code below.\n**Note: If using a shared screening device, ensure the device does not have an\nautofill/auto complete function enabled\n· Please answer each question to complete the voluntary screening.\n· Eligible applicants will be asked to Electronically Sign and click

: 

In [ ]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

: 

In [ ]:
chunks=split_documents(all_pdf_documents)
chunks

Split 9 documents into 24 chunks

Example chunk:
Content: Work Opportunity Tax Credits – KBR, Inc.
KBR, Inc. participates in the Work Opportunity Tax Credit (WOTC) program, which ADP
administers on behalf of the company. Please follow the steps listed below ...
Metadata: {'producer': 'PDF-XChange Printer V6 (6.0 build 317.1) [Windows 10 Enterprise x64 (Build 19045)]', 'creator': 'PyPDF', 'creationdate': '2024-11-25T19:56:12-06:00', 'source': '../data/pdf/Work Opportunity Tax Credit (WOTC).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Work Opportunity Tax Credit (WOTC).pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PDF-XChange Printer V6 (6.0 build 317.1) [Windows 10 Enterprise x64 (Build 19045)]', 'creator': 'PyPDF', 'creationdate': '2024-11-25T19:56:12-06:00', 'source': '../data/pdf/Work Opportunity Tax Credit (WOTC).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Work Opportunity Tax Credit (WOTC).pdf', 'file_type': 'pdf'}, page_content='Work Opportunity Tax Credits – KBR, Inc.\nKBR, Inc. participates in the Work Opportunity Tax Credit (WOTC) program, which ADP\nadministers on behalf of the company. Please follow the steps listed below to screen for\nthe WOTC program.  We appreciate your cooperation.\nApplicant Instructions\n· Open https://tcs.adp.com/kbrinc  or scan the QR code below.\n**Note: If using a shared screening device, ensure the device does not have an\nautofill/auto complete function enabled\n· Please answer each question to complete the voluntary screening.\n· Eligible applicants will be asked to Electronically Sign and click

: 

embedding and vectorstoredb

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

: 

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6759.82it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


: 

### Vector Store


In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


: 

In [ ]:
chunks

[Document(metadata={'producer': 'PDF-XChange Printer V6 (6.0 build 317.1) [Windows 10 Enterprise x64 (Build 19045)]', 'creator': 'PyPDF', 'creationdate': '2024-11-25T19:56:12-06:00', 'source': '../data/pdf/Work Opportunity Tax Credit (WOTC).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Work Opportunity Tax Credit (WOTC).pdf', 'file_type': 'pdf'}, page_content='Work Opportunity Tax Credits – KBR, Inc.\nKBR, Inc. participates in the Work Opportunity Tax Credit (WOTC) program, which ADP\nadministers on behalf of the company. Please follow the steps listed below to screen for\nthe WOTC program.  We appreciate your cooperation.\nApplicant Instructions\n· Open https://tcs.adp.com/kbrinc  or scan the QR code below.\n**Note: If using a shared screening device, ensure the device does not have an\nautofill/auto complete function enabled\n· Please answer each question to complete the voluntary screening.\n· Eligible applicants will be asked to Electronically Sign and click

: 

In [ ]:
###convert texts to embeddings
texts=[doc.page_content for doc in chunks]
texts
###generate embeddings
embeddings=embedding_manager.generate_embeddings(texts)
###store in the vector database
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 24 texts...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.80s/it]

Generated embeddings with shape: (24, 384)
Adding 24 documents to vector store...
Successfully added 24 documents to vector store
Total documents in collection: 24


: 

### RAG Pipeline From Vectorstore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

: 

In [ ]:
rag_retriever

: 

In [ ]:
rag_retriever.retrieve("what shoul you do when you see stop sign")

Retrieving documents for query: 'what shoul you do when you see stop sign'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.12it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_b3304c45_8',
  'content': 'Driving Written Test : \n \n1. When you come to a stop sign, you must stop your vehicle : \n◼ As close to the stop sign as possible. \n◼ At place near the intersection providing you come to a complete stop. \n◼ At marked stop line, before entering the crosswalk, or before entering the \nintersection if there is no crosswalk. \n \n \n2. When headlight are required, they should be dimmed at least 500 feet before \nmeeting and 300 feet before overtaking another vehicle. \n◼ True  \n◼ False \n3. When driving along the highway and the front wheel of your vehicle runs off the \npavement, you should : \n◼ Quickly swing back onto the pavement at your normal speed. \n◼ Grasp the steering wheel tightly & take your foot off the accelerator. \n◼ Apply the brakes immediately and swing the back onto the pavement \nquickly. \n◼  \n4. Most rear end collisions are caused by : \n◼ Dangerous road conditions. \n◼ The vehicle in front stopping too fast. \n◼ The vehic

: 

### Integration VectorDb Context pipeline with LLM output

In [ ]:
### simple RAG pipeline with Groq llm
from langchain_groq import Chatgroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the groq llm (set your GROQ_API_KEY in environment)
os.getenv("GROQ_API_KEY")
llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

ImportError: cannot import name 'Chatgroq' from 'langchain_groq' (/Users/saikumarpadamati/Desktop/RAG/.venv/lib/python3.14/site-packages/langchain_groq/__init__.py)

: 

In [ ]:
answer=rag_simple("what to do when you encounter stop sign?", rag_retriever,llm)
print(answer)

### Enhanced RAG pipeline

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })
        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])


### USING RERANKER STRATEGY

In [ ]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm, rerank_top_n: int = 3):
        self.retriever = retriever
        self.llm = llm
        self.history = []
        self.rerank_top_n = rerank_top_n

    # -----------------------------
    # SIMPLE RERANKER (baseline)
    # -----------------------------
    def rerank(self, query: str, docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        Lightweight reranker:
        combines:
        - original similarity score
        - keyword overlap boost (simple heuristic)
        """

        query_terms = set(query.lower().split())

        def score(doc):
            text = doc['content'].lower()
            overlap = sum(1 for t in query_terms if t in text)

            # combine vector score + lexical match
            return (0.7 * doc['similarity_score']) + (0.3 * overlap / len(query_terms))

        for doc in docs:
            doc['rerank_score'] = score(doc)

        # sort by rerank score
        docs.sort(key=lambda x: x['rerank_score'], reverse=True)

        return docs

    # -----------------------------
    # MAIN QUERY FUNCTION
    # -----------------------------
    def query(self, question: str, top_k: int = 10, min_score: float = 0.2,
              stream: bool = False, summarize: bool = False) -> Dict[str, Any]:

        # STEP 1: Retrieve more candidates
        results = self.retriever.retrieve(
            question,
            top_k=top_k,
            score_threshold=min_score
        )

        if not results:
            return {
                'question': question,
                'answer': "No relevant context found.",
                'sources': [],
                'summary': None,
                'history': self.history
            }

        # STEP 2: RERANK (NEW 🔥)
        reranked_results = self.rerank(question, results)

        # STEP 3: Keep only top N after reranking
        final_results = reranked_results[:self.rerank_top_n]

        # STEP 4: Build context
        context = "\n\n".join([doc['content'] for doc in final_results])

        # STEP 5: Sources (now based on reranked results)
        sources = [{
            'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
            'page': doc['metadata'].get('page', 'unknown'),
            'vector_score': doc['similarity_score'],
            'rerank_score': doc.get('rerank_score', 0),
            'preview': doc['content'][:120] + '...'
        } for doc in final_results]

        # STEP 6: Prompt LLM
        prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question: {question}

Answer:
"""

        response = self.llm.invoke([prompt])
        answer = response.content

        # STEP 7: Citations
        citations = [
            f"[{i+1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]

        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations)

        # STEP 8: Summary (optional)
        summary = None
        if summarize:
            summary_prompt = f"Summarize in 2 sentences:\n{answer}"
            summary = self.llm.invoke([summary_prompt]).content

        # STEP 9: History
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

### USING RERANKER AND MODERN SEARCH TECHNIQUE(BM@%+VECTOR SEARCH)

In [ ]:
from typing import List, Dict, Any
from rank_bm25 import BM25Okapi
import numpy as np
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm, documents: List[Dict], rerank_top_n: int = 3):
        self.retriever = retriever
        self.llm = llm
        self.history = []
        self.rerank_top_n = rerank_top_n

        # -----------------------------
        # BM25 INDEX BUILDING
        # -----------------------------
        self.documents = documents
        self.corpus = [doc['content'].lower().split() for doc in documents]
        self.bm25 = BM25Okapi(self.corpus)

    # -----------------------------
    # BM25 SEARCH
    # -----------------------------
    def bm25_search(self, query: str, top_k: int = 5):
        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)

        top_indices = np.argsort(scores)[::-1][:top_k]

        results = []
        for idx in top_indices:
            doc = self.documents[idx].copy()
            doc['bm25_score'] = float(scores[idx])
            results.append(doc)

        return results

    # -----------------------------
    # VECTOR SEARCH
    # -----------------------------
    def vector_search(self, query: str, top_k: int = 5):
        results = self.retriever.retrieve(query, top_k=top_k)
        return results

    # -----------------------------
    # HYBRID MERGE
    # -----------------------------
    def merge_results(self, bm25_results, vector_results):
        merged = {}

        def add_docs(docs, weight):
            for d in docs:
                key = d['content'][:100]  # simple dedup key

                if key not in merged:
                    merged[key] = d
                    merged[key]['hybrid_score'] = 0

                merged[key]['hybrid_score'] += weight

                if 'bm25_score' in d:
                    merged[key]['bm25_score'] = d['bm25_score']
                if 'similarity_score' in d:
                    merged[key]['vector_score'] = d['similarity_score']

        add_docs(bm25_results, weight=0.4)
        add_docs(vector_results, weight=0.6)

        return list(merged.values())

    # -----------------------------
    # RERANKER
    # -----------------------------
    def rerank(self, query: str, docs: List[Dict]) -> List[Dict]:

        query_terms = set(query.lower().split())

        def score(doc):
            text = doc['content'].lower()
            overlap = sum(1 for t in query_terms if t in text)

            vec = doc.get('vector_score', 0)
            bm25 = doc.get('bm25_score', 0)

            return (0.5 * vec) + (0.3 * bm25) + (0.2 * overlap)

        for doc in docs:
            doc['rerank_score'] = score(doc)

        docs.sort(key=lambda x: x['rerank_score'], reverse=True)
        return docs

    # -----------------------------
    # MAIN QUERY
    # -----------------------------
    def query(self, question: str, top_k: int = 5, min_score: float = 0.2):

        # STEP 1: Retrieve from both systems
        vector_results = self.vector_search(question, top_k=top_k)
        bm25_results = self.bm25_search(question, top_k=top_k)

        # STEP 2: Merge
        merged = self.merge_results(bm25_results, vector_results)

        # STEP 3: Rerank
        reranked = self.rerank(question, merged)

        final_docs = reranked[:self.rerank_top_n]

        # STEP 4: Build context
        context = "\n\n".join([d['content'] for d in final_docs])

        # STEP 5: LLM prompt
        prompt = f"""
Use the following context to answer the question:

{context}

Question: {question}

Answer:
"""

        response = self.llm.invoke([prompt])
        answer = response.content

        # STEP 6: Sources
        sources = [{
            'source': d['metadata'].get('source_file', 'unknown'),
            'page': d['metadata'].get('page', 'unknown'),
            'vector_score': d.get('vector_score', 0),
            'bm25_score': d.get('bm25_score', 0),
            'rerank_score': d.get('rerank_score', 0),
        } for d in final_docs]

        return {
            'answer': answer,
            'sources': sources,
            'context_used': context
        }